# 02 — Data Merge

Merges the four cleaned datasets + the simulator output into a single
master dataset for ML.

**Inputs:**
- `data/interim/clima_limpio.csv`
- `data/interim/energia_limpio.csv`
- `data/interim/ocupacio_aules_horaria.csv`
- `data/raw/Calendari-UPF-2024.csv`
- `data/interim/ocupacion_por_hora.csv` (output of the Repast simulator)

**Output:**
- `data/processed/dataset_smart_campus_master.csv`


In [50]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_RAW = Path('../data/raw')
DATA_INTERIM = Path('../data/interim')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


## 1. Load all cleaned datasets

In [51]:
df_clima = pd.read_csv(DATA_INTERIM / 'clima_limpio.csv', parse_dates=['Timestamp'])
df_energia = pd.read_csv(DATA_INTERIM / 'energia_limpio.csv', parse_dates=['Timestamp'])
df_aulas = pd.read_csv(DATA_INTERIM / 'ocupacio_aules_horaria.csv', parse_dates=['Timestamp'])
df_calendari = pd.read_csv(DATA_RAW / 'Calendari-UPF-2024.csv', parse_dates=['fecha'])
df_simulacion = pd.read_csv(DATA_INTERIM / 'ocupacion_por_hora.csv', parse_dates=['Timestamp'])

print("All datasets loaded:")
for name, df in [('clima', df_clima), ('energia', df_energia),
                 ('aulas', df_aulas), ('calendari', df_calendari),
                 ('simulacion', df_simulacion)]:
    print(f"  {name:12} rows={len(df):>6,}  cols={list(df.columns)}")


All datasets loaded:
  clima        rows= 8,784  cols=['Timestamp', 'Temperatura', 'Lluvia']
  energia      rows= 8,784  cols=['Timestamp', 'Consumo_kWh']
  aulas        rows= 8,784  cols=['Timestamp', 'Aules_Ocupades', 'Ocupacio_Percent']
  calendari    rows=   366  cols=['fecha', 'tipus_dia']
  simulacion   rows= 8,784  cols=['Timestamp', 'Personas_Reales']


## 2. Master merge — energy as base

In [52]:
# Energy is our target → use it as the base for LEFT JOINs
df_final = (
    df_energia
    .merge(df_clima, on='Timestamp', how='left')
    .merge(df_aulas, on='Timestamp', how='left')
)

# Hours without classroom reservations → 0% occupancy (not NaN)
df_final['Aules_Ocupades'] = df_final['Aules_Ocupades'].fillna(0)
df_final['Ocupacio_Percent'] = df_final['Ocupacio_Percent'].fillna(0)

print(f"After merge with weather + classrooms: {df_final.shape}")
df_final.head()


After merge with weather + classrooms: (8784, 6)


,Timestamp,Consumo_kWh,Temperatura,Lluvia,Aules_Ocupades,Ocupacio_Percent
0,2024-01-01 01:00:00,96,8.0,0.0,0.0,0.0
1,2024-01-01 02:00:00,96,7.7,0.0,0.0,0.0
2,2024-01-01 03:00:00,97,7.5,0.0,0.0,0.0
3,2024-01-01 04:00:00,96,7.7,0.0,0.0,0.0
4,2024-01-01 05:00:00,95,7.6,0.0,0.0,0.0


## 3. Add academic calendar (day type)

In [53]:
# The calendar is daily, but our dataset is hourly.
# We need to broadcast the calendar info to every hour of each day.

# Build a 'date' bridge column from the timestamp
df_final['Data_Pont'] = df_final['Timestamp'].dt.normalize()

df_final = pd.merge(
    df_final,
    df_calendari,
    left_on='Data_Pont',
    right_on='fecha',
    how='left'
)

# Cleanup: drop the bridge columns
df_final = df_final.drop(columns=['Data_Pont', 'fecha'])

# Defensive fill: any unlabeled day → assume 'Classe' (lectiu)
df_final['tipus_dia'] = df_final['tipus_dia'].fillna('Classe')

print(f"✅ Calendar integrated.")
print(f"Day-type distribution in master dataset:\n{df_final['tipus_dia'].value_counts()}")
df_final.head(24)

✅ Calendar integrated.
Day-type distribution in master dataset:
tipus_dia
Classe           3362
Altre            1320
Dissabte         1248
Diumenge         1248
Avaluacio         720
Vacances          454
Festiu            192
No lectiu         144
La Benvinguda      96
Name: count, dtype: int64


,Timestamp,Consumo_kWh,Temperatura,Lluvia,Aules_Ocupades,Ocupacio_Percent,tipus_dia
0,2024-01-01 01:00:00,96,8.0,0.0,0.0,0.0,Vacances
1,2024-01-01 02:00:00,96,7.7,0.0,0.0,0.0,Vacances
2,2024-01-01 03:00:00,97,7.5,0.0,0.0,0.0,Vacances
3,2024-01-01 04:00:00,96,7.7,0.0,0.0,0.0,Vacances
4,2024-01-01 05:00:00,95,7.6,0.0,0.0,0.0,Vacances
5,2024-01-01 06:00:00,106,7.2,0.0,0.0,0.0,Vacances
6,2024-01-01 07:00:00,97,7.5,0.0,0.0,0.0,Vacances
7,2024-01-01 08:00:00,98,7.7,0.0,0.0,0.0,Vacances
8,2024-01-01 09:00:00,103,8.2,0.0,0.0,0.0,Vacances
9,2024-01-01 10:00:00,100,8.7,0.0,0.0,0.0,Vacances


## 4. Add simulator-derived Wi-Fi occupancy


In [54]:
# Day of week (numeric + text)
df_final['Dia_Semana_Num'] = df_final['Timestamp'].dt.dayofweek
dias_map = {0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 3: 'Jueves',
            4: 'Viernes', 5: 'Sábado', 6: 'Domingo'}
df_final['Dia_Semana'] = df_final['Dia_Semana_Num'].map(dias_map)

# DEFENSIVE: drop any pre-existing 'Ocupacion_Simulada' column to avoid
# pandas creating an 'Ocupacion_Simulada.1' suffix on merge
if 'Ocupacion_Simulada' in df_final.columns:
    df_final = df_final.drop(columns=['Ocupacion_Simulada'])
    print("⚠️  Pre-existing 'Ocupacion_Simulada' dropped before merge.")

# Merge with simulator output
df_final = pd.merge(
    df_final,
    df_simulacion[['Timestamp', 'Personas_Reales']],
    on='Timestamp',
    how='left'
)
df_final = df_final.rename(columns={'Personas_Reales': 'Ocupacion_Simulada'})
df_final['Ocupacion_Simulada'] = df_final['Ocupacion_Simulada'].fillna(0)

# VERIFICATION: no '.1' columns should exist
duplicated_cols = [c for c in df_final.columns if c.endswith('.1')]
assert len(duplicated_cols) == 0, f"❌ Duplicated columns found: {duplicated_cols}"
print("✅ No duplicated columns. Merge clean.")


✅ No duplicated columns. Merge clean.


## 5. Repair missing values intelligently

In [55]:
# Diagnostic: where are the NaN?
print("--- Null count per column ---")
nulos = df_final.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "(no nulls)")

# Repair strategy:
# - Rain: NaN means "no rain" → 0
# - Temperature: missing hours → linear interpolation between neighbors
df_final['Lluvia'] = df_final['Lluvia'].fillna(0.0)
df_final['Temperatura'] = df_final['Temperatura'].interpolate()

# Final cleanup of any residual NaN
n_before = len(df_final)
df_final = df_final.dropna()
print(f"\nDropped {n_before - len(df_final)} residual NaN rows.")
print(f"Final shape: {df_final.shape}")


--- Null count per column ---
Temperatura      1
Lluvia         252
dtype: int64

Dropped 0 residual NaN rows.
Final shape: (8784, 10)


## 6. Save master dataset

In [56]:
out_path = DATA_PROCESSED / 'dataset_smart_campus_master.csv'
df_final.to_csv(out_path, index=False)
print(f"✅ Saved master dataset to {out_path}")
print(f"Final columns: {list(df_final.columns)}")
df_final.head()


✅ Saved master dataset to ..\data\processed\dataset_smart_campus_master.csv
Final columns: ['Timestamp', 'Consumo_kWh', 'Temperatura', 'Lluvia', 'Aules_Ocupades', 'Ocupacio_Percent', 'tipus_dia', 'Dia_Semana_Num', 'Dia_Semana', 'Ocupacion_Simulada']


,Timestamp,Consumo_kWh,Temperatura,Lluvia,Aules_Ocupades,Ocupacio_Percent,tipus_dia,Dia_Semana_Num,Dia_Semana,Ocupacion_Simulada
0,2024-01-01 01:00:00,96,8.0,0.0,0.0,0.0,Vacances,0,Lunes,0.0
1,2024-01-01 02:00:00,96,7.7,0.0,0.0,0.0,Vacances,0,Lunes,0.0
2,2024-01-01 03:00:00,97,7.5,0.0,0.0,0.0,Vacances,0,Lunes,0.0
3,2024-01-01 04:00:00,96,7.7,0.0,0.0,0.0,Vacances,0,Lunes,0.0
4,2024-01-01 05:00:00,95,7.6,0.0,0.0,0.0,Vacances,0,Lunes,0.0
